# 04 — Production Forecasting (Arps Decline Curve Analysis)
## Well Production Forecasting & Performance Analytics System

**Decline Curve Analysis Models:**
- **Exponential** (b=0): `q(t) = Qi · exp(−Di · t)` — fastest decline, conservative
- **Hyperbolic** (0<b<1): `q(t) = Qi / (1 + b·Di·t)^(1/b)` — most realistic, industry standard
- **Harmonic** (b=1): `q(t) = Qi / (1 + Di·t)` — slowest decline, optimistic

Best model selected by R² score. Forecasts generated at:
- **Well level** (top 100 wells by cumulative production)
- **Operator level** (all 18 operators)
- **Basin level** (Bakken, Permian, Eagle Ford)

24-month forward projection from Dec 2023 → Dec 2025.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pandas as pd
import numpy as np

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .master('local[*]')
    .appName('WellAnalytics-Forecast')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.databricks.delta.schema.autoMerge.enabled', 'true')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.driver.memory', '4g')
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel('WARN')

sns.set_theme(style='whitegrid', palette='muted')
PLOT_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data', 'plots')
os.makedirs(PLOT_DIR, exist_ok=True)
print(f'Spark {spark.version} ready')

## Step 1 — Run Forecasting (Fit Arps + Write Forecast Table)

In [ ]:
from src.forecasting import run_forecasting

total_rows = run_forecasting(spark, n_months=24)
print(f'\nTotal forecast rows written: {total_rows:,}')

## Step 2 — Demonstrate Arps Curve Fitting on Sample Wells

In [ ]:
from pyspark.sql import functions as F
from src.spark_session import silver_path, gold_path
from src.forecasting import fit_arps, project_forward, arps_hyperbolic

prod = spark.read.format('delta').load(silver_path('silver_production'))

# Get top 3 wells by cumulative oil for showcase
top3 = (
    prod.filter(F.col('oil_and_gas_group') == 'O')
    .groupBy('api_number', 'well_name')
    .agg(F.sum('production').alias('cum_oil'))
    .orderBy(F.col('cum_oil').desc())
    .limit(3)
    .toPandas()
)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, row in top3.iterrows():
    api = row['api_number']
    well_label = (row['well_name'] or api)[:20]

    hist = (
        prod.filter((F.col('api_number') == api) & (F.col('oil_and_gas_group') == 'O'))
        .orderBy('production_month')
        .toPandas()
    )
    if len(hist) < 6:
        continue

    t = np.arange(len(hist), dtype=float)
    q = hist['production'].values
    params = fit_arps(t, q)

    # Reconstruct fitted curve
    from src.forecasting import _exp, _hyp, _har
    if params['model'] == 'exponential':
        fitted = _exp(t, params['qi'], params['di'])
    elif params['model'] == 'harmonic':
        fitted = _har(t, params['qi'], params['di'])
    else:
        fitted = _hyp(t, params['qi'], params['di'], params['b'])

    # Forecast
    forecast = project_forward(params, int(t[-1]), n_months=24)
    t_future = np.arange(len(t), len(t) + 24, dtype=float)

    ax = axes[i]
    ax.scatter(t, q, s=20, color='steelblue', alpha=0.6, label='Historical')
    ax.plot(t, fitted, color='darkorange', linewidth=2, label=f'Fit ({params["model"]}, R²={params["r2"]:.3f})')
    ax.plot(t_future, forecast, color='green', linewidth=2, linestyle='--', label='Forecast (24 mo)')
    ax.axvline(x=t[-1], color='gray', linestyle=':', linewidth=1.5)
    ax.set_title(f'API: ...{api[-6:]}\n{well_label}', fontsize=10)
    ax.set_xlabel('Month #')
    ax.set_ylabel('Oil Production (BBL)')
    ax.legend(fontsize=8)

plt.suptitle('Arps Decline Curve Fit + 24-Month Forecast (Top 3 Wells)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, '06_decline_curve_wells.png'), dpi=150)
plt.show()
print('Saved: 06_decline_curve_wells.png')

## Step 3 — Basin-Level Historical vs. Forecast Comparison

In [ ]:
forecast_df = spark.read.format('delta').load(gold_path('gold_production_forecast')).toPandas()
forecast_df['forecast_month'] = pd.to_datetime(forecast_df['forecast_month'])

prod_pdf = (
    prod.filter(F.col('oil_and_gas_group') == 'O')
    .groupBy('shale_play', 'production_month')
    .agg(F.sum('production').alias('total_oil'))
    .toPandas()
)
prod_pdf['production_month'] = pd.to_datetime(prod_pdf['production_month'])

basin_fcst = forecast_df[forecast_df['entity_type'] == 'basin']

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)

basins = prod_pdf['shale_play'].unique()[:3]
for idx, basin in enumerate(basins):
    hist = prod_pdf[prod_pdf['shale_play'] == basin].sort_values('production_month')
    fcst = basin_fcst[basin_fcst['entity_id'] == basin].sort_values('forecast_month')

    ax = axes[idx]
    ax.plot(hist['production_month'], hist['total_oil'] / 1e3,
            color='steelblue', linewidth=2, label='Historical')
    if len(fcst) > 0:
        ax.plot(fcst['forecast_month'], fcst['forecast_oil_bbl'] / 1e3,
                color='green', linewidth=2, linestyle='--', label='Forecast')
        # Confidence band ±15%
        ax.fill_between(
            fcst['forecast_month'],
            fcst['forecast_oil_bbl'] * 0.85 / 1e3,
            fcst['forecast_oil_bbl'] * 1.15 / 1e3,
            alpha=0.2, color='green', label='±15% band'
        )
    ax.set_title(basin.replace(', US', ''), fontsize=11, fontweight='bold')
    ax.set_xlabel('Month')
    ax.set_ylabel('Oil Production (K BBL)')
    ax.legend(fontsize=9)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.suptitle('Historical vs. Forecast — Basin-Level Oil Production', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, '07_basin_forecast.png'), dpi=150)
plt.show()
print('Saved: 07_basin_forecast.png')

## Step 4 — R² Score Distribution (Model Quality)

In [ ]:
well_fcst = forecast_df[forecast_df['entity_type'] == 'well'].drop_duplicates(subset=['entity_id'])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# R² distribution
ax = axes[0]
ax.hist(well_fcst['r2_score'].dropna(), bins=30, color='steelblue', edgecolor='white')
ax.axvline(x=well_fcst['r2_score'].mean(), color='red', linestyle='--',
           label=f'Mean R² = {well_fcst["r2_score"].mean():.3f}')
ax.set_xlabel('R² Score', fontsize=12)
ax.set_ylabel('Number of Wells', fontsize=12)
ax.set_title('Arps Fit Quality — R² Distribution (Well Level)', fontsize=12, fontweight='bold')
ax.legend(fontsize=11)

# Model type breakdown
ax2 = axes[1]
model_counts = well_fcst['model_type'].value_counts()
ax2.bar(model_counts.index, model_counts.values,
        color=['#3498db', '#e67e22', '#2ecc71'][:len(model_counts)])
ax2.set_xlabel('Decline Model', fontsize=12)
ax2.set_ylabel('Number of Wells', fontsize=12)
ax2.set_title('Best-Fit Model Selection by Well', fontsize=12, fontweight='bold')
for bar in ax2.patches:
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             str(int(bar.get_height())), ha='center', fontsize=12)

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, '08_model_quality.png'), dpi=150)
plt.show()
print('Saved: 08_model_quality.png')

## Step 5 — Operator Forecast Summary (Strategic Planning)

In [ ]:
op_fcst = (
    forecast_df[forecast_df['entity_type'] == 'operator']
    .groupby('entity_id')
    .agg(forecast_oil_bbl=('forecast_oil_bbl', 'sum'))
    .reset_index()
    .sort_values('forecast_oil_bbl', ascending=True)
    .tail(12)
)

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(op_fcst['entity_id'], op_fcst['forecast_oil_bbl'] / 1e6,
               color=sns.color_palette('Greens_r', len(op_fcst)))
ax.set_xlabel('Forecast Total Oil (Million BBL, 24 months)', fontsize=12)
ax.set_title('24-Month Oil Production Forecast by Operator', fontsize=14, fontweight='bold')
for bar in bars:
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.1f}M', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, '09_operator_forecast.png'), dpi=150)
plt.show()
print('Saved: 09_operator_forecast.png')

print(f'\nAll plots saved to: {PLOT_DIR}')
print(os.listdir(PLOT_DIR))